# spectral-change: setup, tests, demo

Keep this notebook in the same folder as `spectral-change/` (e.g. `agentic-eo/test/`). The repo root with `.env` and `lib/` is found automatically. Run with the **Agentic EO** kernel, top to bottom.

Tip: add `CHANGE_MAPS_DIR=/home/sagemaker-user/agentic-eo/test/outputs` to `.env` so maps, plots and CSVs survive a restart and show up in the file browser (default is `/tmp/spectral_change`).

In [ ]:
# 1. Install the extra packages into THIS kernel's Python (the servers run with it).
#    The lab kernel is a uv environment, so use uv first and fall back to pip.
import shutil
import subprocess
import sys

print("kernel python:", sys.executable)
pkgs = ["spyndex>=0.6.0", "tifffile>=2023.7.0", "matplotlib>=3.8.0"]
uv = shutil.which("uv")
cmd = [uv, "pip", "install", "--python", sys.executable, *pkgs] if uv else [sys.executable, "-m", "pip", "install", *pkgs]
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:])
check = subprocess.run([sys.executable, "-c", "import spyndex, tifffile, matplotlib; print('OK', len(spyndex.indices), 'indices')"],
                       capture_output=True, text=True)
print(check.stdout, check.stderr[-500:])

In [ ]:
# 2. Paths and credentials check (values are not printed)
import json
import os
import pathlib

from dotenv import load_dotenv
from IPython.display import Image, display

HERE = pathlib.Path.cwd()  # folder that contains spectral-change/
ROOT = next(p for p in [HERE, *HERE.parents] if (p / 'lib' / 'agentic_eo').exists())
SERVER = next(HERE.rglob('spectral-change/server.py'))
assert SERVER.exists(), 'open this notebook next to the spectral-change folder'
load_dotenv(ROOT / '.env')
print('repo root:', ROOT, '| server:', SERVER)
print('CDSE_CLIENT_ID set:', bool(os.getenv('CDSE_CLIENT_ID')))
print('CDSE_CLIENT_SECRET set:', bool(os.getenv('CDSE_CLIENT_SECRET')))
print('outputs go to:', os.getenv('CHANGE_MAPS_DIR', '/tmp/spectral_change'))
print('node available (JS checks):', shutil.which('node') is not None)

In [ ]:
# 3. Offline tests (expect 50/50)
r = subprocess.run([sys.executable, 'test.py'], cwd=SERVER.parent, capture_output=True, text=True)
print(r.stdout[-6000:])
if r.returncode:
    print(r.stderr[-1500:])

In [ ]:
# 4. Real CDSE calls: change NDBI, time series NDVI (whole box), change NDVI + focused time series on its decrease_extent
r = subprocess.run([sys.executable, 'test.py', '--live'], cwd=SERVER.parent, capture_output=True, text=True)
print(r.stdout[-8000:])
if r.returncode:
    print(r.stderr[-2000:])

## Call the tools directly (no agent) and look at the figures

In [ ]:
# 5. How much changed: map before | after | delta | change classes
for _m in ('server', 'asi_formula', 'plots'):  # drop modules cached from the old two-server version
    sys.modules.pop(_m, None)
sys.path.insert(0, str(SERVER.parent))
import server as sc  # noqa: E402

BBOX = '13.76,52.38,13.82,52.41'  # Tesla Gigafactory, Grünheide
change = json.loads(await sc.compute_index_change(
    bbox=BBOX, index='NDVI',
    before_start='2019-06-01', before_end='2019-08-31',
    after_start='2024-06-01', after_end='2024-08-31',
))
print({k: change.get(k) for k in ('ok', 'increase_ha', 'decrease_ha', 'decrease_extent', 'valid_fraction', 'error')})
if change.get('map_path'):
    display(Image(change['map_path'], width=900))

In [ ]:
# 6. When did it change: time series over the whole box vs. only the changed area (decrease_extent)
for label, box in [('whole box', BBOX), ('changed area', change.get('decrease_extent'))]:
    if not box:
        continue
    ts = json.loads(await sc.compute_index_timeseries(bbox=box, index='NDVI', start_date='2018-01-01', end_date='2024-12-31'))
    print(label, {k: ts.get(k) for k in ('ok', 'bbox', 'scenes_used', 'change_point', 'trend_per_year', 'error')})
    if ts.get('plot_path'):
        display(Image(ts['plot_path'], width=1000))

## Demo with the EVE agent

In [ ]:
# 7. Agent: spectral-change (local, stdio) + registry geocode
sys.path.insert(0, str(ROOT / 'lib'))
from agentic_eo import create_agent, show_trace  # noqa: E402

SERVERS = [str(SERVER), 'geocode']
prompt = (
    'You are an Earth Observation analyst. Answer only with numbers that come from tool results, '
    'and say which tool each number came from. '
    '1) If the user gives a place name instead of coordinates, call geocode_place to get the bbox. '
    '2) Call search_spectral_indices to pick the index that matches what the user asks about '
    '(built-up -> NDBI, vegetation -> NDVI, water -> MNDWI). '
    '3) For how much changed between two periods, call compute_index_change with the same season in both '
    'periods (e.g. June to August of each year). Report increase_ha AND decrease_ha, the mean change, '
    'the periods, the index name and reference, valid_fraction and warnings. Give map_path as plain text. '
    '4) For when something changed, call compute_index_timeseries. If compute_index_change was already run '
    'for this area, use its decrease_extent (for losses) or increase_extent (for gains) as the bbox. '
    'Report change_point, yearly medians and trend_per_year, and give plot_path as plain text. '
    'If a tool returns ok=false, say what failed.'
)
agent = await create_agent(SERVERS, prompt=prompt)
print('tools:', agent.tool_names)
try:
    show_trace(await agent.chat(
        'How much built-up area was added at the Tesla Gigafactory near Grünheide, Germany '
        'between summer 2019 and summer 2024? Area: 13.76,52.38,13.82,52.41.'))
    show_trace(await agent.chat('And how much vegetation was lost in the same area and periods?'))
    show_trace(await agent.chat('When did that vegetation loss happen? Look at NDVI from 2018 to 2024.'))
finally:
    await agent.close()

In [ ]:
# 8. Show the newest figures (the agent's server writes to the same folder)
import glob  # noqa: E402

for pattern in ('*_change.png', '*_timeseries.png'):
    files = sorted(glob.glob(str(sc.DEFAULT_SAVE_DIR / pattern)), key=os.path.getmtime)
    if files:
        print(files[-1])
        display(Image(files[-1], width=900))